# Tunda Box assessment — reproducible calculations

All data is invented for the assessment. Assumptions: remove the exact duplicate order row (1006); count only completed orders for completed revenue; pandas sums skip null amounts; retain the KES 185,000 amount pending verification; normalize city whitespace/case; and keep unmatched customers visible. The repeat-buyer denominator is all registered customers, and repeat buyers have at least two distinct completed order IDs.

In [ ]:
from math import erf, sqrt

import pandas as pd

In [ ]:
customers = pd.DataFrame([
    (101, "Nairobi", "2026-01-14", "Organic"),
    (102, "nairobi", "2026-02-03", "Referral"),
    (103, "Mombasa", "2026-06-20", "Paid Social"),
    (104, "Kisumu", "2026-03-02", "Organic"),
    (105, "Nairobi ", "2026-03-15", "Paid Social"),
    (106, "Nakuru", "2026-04-01", "Referral"),
    (107, "Mombasa", "2026-04-18", "Organic"),
    (108, "Nairobi", "2026-05-09", "Paid Social"),
    (109, "Eldoret", "2026-06-21", "Organic"),
    (110, "Kisumu", "2026-07-05", "Paid Social"),
], columns=["customer_id", "city", "signup_date", "channel"])

orders = pd.DataFrame([
    (1001, 101, "2026-06-03", "completed", 2400, "M-Pesa"),
    (1002, 102, "2026-06-05", "completed", 1850, "M-Pesa"),
    (1003, 103, "2026-06-11", "cancelled", 3200, "Card"),
    (1004, 104, "2026-06-18", "completed", 2100, "M-Pesa"),
    (1005, 101, "2026-06-25", "completed", 2650, "Card"),
    (1006, 105, "2026-07-02", "completed", 1500, "M-Pesa"),
    (1006, 105, "2026-07-02", "completed", 1500, "M-Pesa"),
    (1007, 106, "2026-07-09", "refunded", 2900, "Card"),
    (1008, 102, "2026-07-14", "completed", 3100, "M-Pesa"),
    (1009, 107, "2026-07-21", "completed", None, "M-Pesa"),
    (1010, 101, "2026-07-28", "completed", 2200, "M-Pesa"),
    (1011, 108, "2026-08-04", "completed", 1750, "Card"),
    (1012, 103, "2026-08-09", "completed", 2800, "M-Pesa"),
    (1013, 104, "2026-08-15", "completed", 185000, "M-Pesa"),
    (1014, 105, "2026-08-19", "cancelled", 1600, "M-Pesa"),
    (1015, 102, "2026-08-27", "completed", 2450, "M-Pesa"),
    (1016, 106, "2026-09-03", "completed", 3300, "Card"),
    (1017, 101, "2026-09-08", "completed", 2750, "M-Pesa"),
    (1018, 108, "2026-09-14", "refunded", 2000, "M-Pesa"),
    (1019, 107, "2026-09-22", "completed", 2350, "Card"),
    (1020, 999, "2026-09-26", "completed", 1900, "M-Pesa"),
], columns=["order_id", "customer_id", "order_date", "status", "amount_kes", "payment_method"])

orders["order_date"] = pd.to_datetime(orders["order_date"])
orders_clean = orders.drop_duplicates().copy()
assert len(orders) == 21 and len(orders_clean) == 20
orders_clean.head()

## Orders by status

Count unique order IDs rather than input rows, so the repeated row for order 1006 is not counted twice.

In [ ]:
status_counts = (orders_clean.groupby("status")["order_id"]
                 .nunique().rename("order_count").sort_index())
print(status_counts)
assert status_counts.to_dict() == {"cancelled": 2, "completed": 16, "refunded": 2}

## Completed revenue by city

Normalize city case and surrounding whitespace. An unmatched customer remains visible as `Unmatched`; it is not assigned to a city.

In [ ]:
customers_clean = customers.copy()
customers_clean["city_clean"] = customers_clean["city"].str.strip().str.title()
city_orders = orders_clean.merge(
    customers_clean[["customer_id", "city_clean"]], on="customer_id", how="left"
)
city_orders["city_clean"] = city_orders["city_clean"].fillna("Unmatched")
completed = city_orders.loc[city_orders["status"].eq("completed")]
city_revenue = (completed.groupby("city_clean")["amount_kes"]
                .sum().sort_values(ascending=False).rename("revenue_kes"))
print(city_revenue)
assert city_revenue.iloc[0] == 187100
assert city_revenue.index[0] == "Kisumu"

## Customers who never ordered and repeat purchase rate

For never-ordered customers, any order status counts as having placed an order. For repeat buying, use distinct completed order IDs and all registered customers as the denominator.

In [ ]:
customers_with_orders = set(orders_clean["customer_id"])
never_ordered = customers.loc[~customers["customer_id"].isin(customers_with_orders), "customer_id"]
print("Never ordered:", never_ordered.tolist())

completed_per_customer = (orders_clean.loc[orders_clean["status"].eq("completed")]
                          .groupby("customer_id")["order_id"].nunique())
repeat_buyer_ids = completed_per_customer[completed_per_customer.ge(2)].index.tolist()
repeat_rate_pct = 100 * len(repeat_buyer_ids) / len(customers)
print("Repeat buyers:", repeat_buyer_ids)
print(f"Repeat purchase rate: {repeat_rate_pct:.1f}% ({len(repeat_buyer_ids)}/{len(customers)})")
assert never_ordered.tolist() == [109, 110]
assert repeat_buyer_ids == [101, 102, 104, 107]
assert repeat_rate_pct == 40

## Month-on-month completed revenue

Null amounts are skipped by the sum. All months with at least one completed order are represented; a calendar table would be needed to display months with no orders.

In [ ]:
monthly_revenue = (orders_clean.loc[orders_clean["status"].eq("completed")]
                   .set_index("order_date")["amount_kes"]
                   .resample("MS").sum().rename("revenue_kes").to_frame())
monthly_revenue["pct_change"] = monthly_revenue["revenue_kes"].pct_change() * 100
print(monthly_revenue.round(1))
assert monthly_revenue["revenue_kes"].tolist() == [9000, 6800, 192000, 10300]

## A/B test

Compute a pooled two-proportion z-test under the null of equal conversion rates. The p-value is two-sided.

In [ ]:
current_users, current_conversions = 1000, 50
new_users, new_conversions = 1000, 60
current_rate = current_conversions / current_users
new_rate = new_conversions / new_users
pooled_rate = (current_conversions + new_conversions) / (current_users + new_users)
standard_error = sqrt(pooled_rate * (1 - pooled_rate) * (1/current_users + 1/new_users))
z_score = (new_rate - current_rate) / standard_error
normal_cdf = lambda x: (1 + erf(x / sqrt(2))) / 2
p_value = 2 * (1 - normal_cdf(abs(z_score)))
print(f"Current: {current_rate:.1%}; new: {new_rate:.1%}")
print(f"Absolute change: {(new_rate-current_rate)*100:.1f} percentage point")
print(f"Relative change: {(new_rate/current_rate-1)*100:.1f}%")
print(f"z = {z_score:.3f}; two-sided p = {p_value:.3f}")
assert abs(p_value - 0.327) < 0.01

## Channel conversion and traffic mix

The aggregate conversion changes as the traffic mix shifts, even when within-channel rates do not fall.

In [ ]:
traffic = pd.DataFrame({
    "channel": ["Organic", "Paid Social"],
    "sep_sessions": [40000, 10000],
    "sep_orders": [2400, 200],
    "oct_sessions": [38000, 30000],
    "oct_orders": [2318, 630],
})
traffic["sep_conversion_pct"] = 100 * traffic["sep_orders"] / traffic["sep_sessions"]
traffic["oct_conversion_pct"] = 100 * traffic["oct_orders"] / traffic["oct_sessions"]
traffic["sep_session_mix_pct"] = 100 * traffic["sep_sessions"] / traffic["sep_sessions"].sum()
traffic["oct_session_mix_pct"] = 100 * traffic["oct_sessions"] / traffic["oct_sessions"].sum()
print(traffic.round(2).to_string(index=False))
sep_total = traffic["sep_orders"].sum() / traffic["sep_sessions"].sum()
oct_total = traffic["oct_orders"].sum() / traffic["oct_sessions"].sum()
print(f"Blended conversion: Sep {sep_total:.2%}; Oct {oct_total:.2%}")
assert round(sep_total * 100, 1) == 5.2
assert round(oct_total * 100, 1) == 4.3

## Typical completed order value

Calculate the mean and median on distinct completed orders with known amounts. The median is less sensitive to the KES 185,000 outlier.

In [ ]:
valid_completed_amounts = orders_clean.loc[
    orders_clean["status"].eq("completed"), "amount_kes"
].dropna()
print(f"Valid completed order values: {len(valid_completed_amounts)}")
print(f"Mean: KES {valid_completed_amounts.mean():,.2f}")
print(f"Median: KES {valid_completed_amounts.median():,.2f}")
assert len(valid_completed_amounts) == 15
assert valid_completed_amounts.median() == 2400
assert round(valid_completed_amounts.mean(), 2) == 14540.00

## Spreadsheet formulas

Completed M-Pesa revenue: `=SUMIFS(E:E,D:D,"completed",F:F,"M-Pesa")`

Look up the city on the customers sheet for an order row: `=XLOOKUP(B2,customers!A:A,customers!B:B,"Unknown")`.